In [1]:
import sys
import os
# current_dir = os.path.dirname(os.path.abspath(__file__))
# project_root = os.path.dirname(current_dir)
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
os.chdir('/bozdagpool/UNT/ew0297/scMNB')
if project_root not in sys.path:
    sys.path.append(project_root)

from model.scMNB_model import scMNB 
from model.trainer import scMNB_Trainer 
from model.gene_clustering import GeneClustering
from model.extract_GRN import get_database_information, extract_GRN_network


import scvi 
import numpy as np 
import torch 
import pandas as pd 
import scanpy as sc 
import anndata as ad
import scipy.sparse as sp
import re 




adata = sc.read_h5ad('/bozdagpool/UNT/ew0297/task_grn_inference/resources/grn_benchmark/inference_data/op_rna.h5ad')
min_cell_fraction = 0.01  # 1% of cells
min_cells_required = int(adata.n_obs * min_cell_fraction)
sc.pp.filter_genes(adata, min_cells=min_cells_required)
dataset_id = adata.uns.get('dataset_id', '')
n_genes = adata.n_vars
tf_path = '/bozdagpool/UNT/ew0297/task_grn_inference/resources_test/grn_benchmark/prior/tf_all.csv'
tf_all = set(np.loadtxt(tf_path, dtype=str))
# # --- Define blacklist regular expressions ---
# # 1. Basic blacklist: Mitochondrial (MT-), Ribosomal (RP), Unnamed/Long non-coding (LINC, AC, AL, etc.), Metabolic complexes (ATP5, UQCR)
basic_patterns = [
    r'^MT-', r'^RP[SL]\d+', r'\.', r'-AS\d+$', r'^LINC', 
    r'^AL\d+\.', r'^AC\d+\.', r'^ATP5', r'^UQCR'
]
# 2. Immediate Early Genes & Cell dissociation stress genes
stress_patterns = [
    r'^FOS[BL]?\d*$',      # FOS, FOSB, FOSL1, FOSL2, etc.
    r'^JUN[BD]?$',         # JUN, JUNB, JUND
    r'^EGR\d+$',           # EGR1, EGR2, EGR3, EGR4
    r'^HSP[A-Z]\d+',       # Heat shock protein family (HSPA1A, HSPB1, HSP90AA1, etc.)
    r'^DNAJ[A-Z]\d+',      # Heat shock protein chaperones (DNAJA1, etc.)
    r'^DUSP\d+$',          # Dual-specificity phosphatases (DUSP1, DUSP2, etc., typical stress response genes)
    r'^IER\d+$',           # Immediate early response genes (IER2, IER3)
    r'^DDIT\d+$',          # DNA damage-inducible transcripts (DDIT3, DDIT4)
    r'^ATF[34]$',          # Activating transcription factors 3 and 4 (classic cell stress markers)
    r'^ZFP36.*',           # ZFP36 family (RNA-binding proteins, rapid stress response)
    r'^ZNF\d*',            # ZNF family (Zinc finger proteins, e.g., ZNF1, ZNF2, ZNF36, etc.)
]
# # Combine all blacklist patterns
blacklist_patterns = basic_patterns + stress_patterns
regex = re.compile('|'.join(blacklist_patterns))
# # 1. Generate boolean mask (True represents blacklist genes, False represents normal genes)
blacklist_mask = np.array([bool(regex.search(g)) for g in adata.var_names])

scMNB_label = np.where(blacklist_mask, -1, 0).tolist()
scMNB_cluster = GeneClustering(adata.var_names , scMNB_label)
scMNB_GRN = scMNB(n_input=n_genes, clustering=scMNB_cluster)


scMNB_GRN_Trainer = scMNB_Trainer(scMNB_GRN, adata, scMNB_cluster, device="cuda:2", 
                                lambda_emp = 1, batch_size=256, lr=1e-3, n_epochs_kl_warmup=20)
scMNB_GRN_Trainer.train(max_epochs=20)
cluster0_gene_list = scMNB_cluster.gene_names[scMNB_cluster.cluster_indices[0]]
scMNB_signed_edges = scMNB_GRN_Trainer.extract_signed_edges_from_omega(0, cluster0_gene_list)

df_grn = extract_GRN_network(adata, 
    scMNB_signed_edges, 
    max_edges=50000,
    distance_threshold=10e6,
    sex=True, 
)

prediction = df_grn[['Source', 'Target', 'Weight']].rename(
    columns={'Source': 'source', 'Target': 'target', 'Weight': 'weight'}
)
prediction = prediction[prediction['source'].isin(tf_all)].reset_index(drop=True)
prediction = prediction.astype(str)
output = ad.AnnData(
    X=None,
    uns={
        'method_id': 'scmnb',
        'dataset_id': dataset_id,
        'prediction': prediction[['source', 'target', 'weight']],
    },
)
output.write_h5ad('/bozdagpool/UNT/ew0297/task_grn_inference/output/GRN_op_result_apr20_1.h5ad', compression='gzip')
import subprocess
command = [
    "bash",
    "/bozdagpool/UNT/ew0297/task_grn_inference/src/metrics/all_metrics/run_local.sh",
    "--dataset", dataset_id,
    "--prediction", "/bozdagpool/UNT/ew0297/task_grn_inference/output/GRN_op_result_apr20_1.h5ad",
    "--score", "/bozdagpool/UNT/ew0297/task_grn_inference/output/op_score_1.h5ad",
    "--num_workers", "20"
]
result = subprocess.run(command, cwd='/bozdagpool/UNT/ew0297/task_grn_inference', capture_output=True, text=True)
print(result.stdout)
score = sc.read_h5ad('/bozdagpool/UNT/ew0297/task_grn_inference/output/op_score_1.h5ad')
df = pd.DataFrame({
    'metric': score.uns['metric_ids'],
    'value':  score.uns['metric_values']
})
df.to_csv('/bozdagpool/UNT/ew0297/scMNB/figure/op_benchmark_result.csv', index=False)

/bozdagpool/UNT/ew0297/MZIPLN/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
                                                                                                                                                                                                         

loading TF list
number of TF-TF edges: 151758
number of TF-Target edges: 5293457


Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed



number of GRN edges in final output: 50000
Layer is set to: lognorm
Regression type is set to: ridge
Number of workers is set to: 20
Dataset is set to: op
Prediction file is set to: /bozdagpool/UNT/ew0297/task_grn_inference/output/GRN_op_result_apr20_1.h5ad
Score file is set to: /bozdagpool/UNT/ew0297/task_grn_inference/output/op_score_1.h5ad
Method id: scmnb, Dataset id: op
Computing metrics for dataset op: ['regression', 'vc', 'rc_tf_act', 'tf_binding', 'sem', 'gs_recovery']
Computing metric: regression
Evaluating 13595 genes (consensus data available for all)
Original net shape:  (42649, 3)
Supplementary columns for grouping: []
Network shape after cleaning: (42649, 3)
Network shape applying max_n_links: (42649, 3)
Static approach (theta=r_precision):
Static approach (theta=r_recall):
Raw approach (no theta):
theta    r2_raw  r_precision  r_recall      r_f1
0      0.511974     0.404443  0.274481  0.327023
Computing metric: vc
Found 192 control samples and 1978 perturbed samples
Ori